# Problema de Transbordo em 3 Camadas — Milho

**Pesquisa Operacional — Gurobi / gurobipy**

Este notebook implementa o modelo de otimização de transporte e armazenamento de milho descrito abaixo:

**Fazendas → Silos → Consumidores**

## 1. Descrição do problema

Uma cooperativa agrícola possui **3 fazendas de milho**, **5 silos intermediários** e **10 consumidores**. O milho é transportado das fazendas para os silos e, em seguida, dos silos para os consumidores. Considere:

- oferta de 1 t, 2 t e 2 t nas fazendas 1, 2 e 3, respectivamente;
- demanda de 0,5 t (500 kg) para cada consumidor;
- capacidade máxima de 1 t em cada silo;
- conservação de fluxo em cada silo;
- limite máximo de 0,5 t em cada rota silo-consumidor;
- custos por tonelada explicitados nas células de dados;
- fluxos contínuos e não negativos;
- minimização do custo total das duas etapas.

A oferta total é 5 t e a demanda total também é 5 t, portanto o problema é balanceado.

## 2. Dados da rede

### Ofertas, capacidades e demandas

| Fazenda | Oferta (t) |
|:---:|---:|
| 1 | 1,0 |
| 2 | 2,0 |
| 3 | 2,0 |

Cada silo possui capacidade máxima de **1,0 t**. Cada consumidor possui demanda de **0,5 t** e cada rota silo-consumidor possui limite máximo de **0,5 t**.

| Consumidor | Demanda (t) |
|:---:|---:|
| 1 | 0,5 |
| 2 | 0,5 |
| 3 | 0,5 |
| 4 | 0,5 |
| 5 | 0,5 |
| 6 | 0,5 |
| 7 | 0,5 |
| 8 | 0,5 |
| 9 | 0,5 |
| 10 | 0,5 |

### Custos por rota: fazenda → silo

Valores em **R$/t**.

| Fazenda / Silo | 1 | 2 | 3 | 4 | 5 |
|:---|---:|---:|---:|---:|---:|
| 1 | 9.000 | 13.000 | 8.000 | 11.000 | 7.000 |
| 2 | 12.000 | 8.000 | 10.000 | 6.000 | 11.000 |
| 3 | 10.000 | 11.000 | 7.000 | 9.000 | 8.000 |

### Custos por rota: silo → consumidor

Valores em **R$/t**.

| Silo / Consumidor | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|:---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 7.000 | 5.000 | 9.000 | 6.000 | 8.000 | 7.000 | 10.000 | 6.000 | 8.000 | 9.000 |
| 2 | 6.000 | 8.000 | 5.000 | 9.000 | 7.000 | 6.000 | 8.000 | 10.000 | 6.000 | 7.000 |
| 3 | 9.000 | 7.000 | 6.000 | 5.000 | 9.000 | 8.000 | 6.000 | 7.000 | 10.000 | 5.000 |
| 4 | 5.000 | 9.000 | 8.000 | 7.000 | 6.000 | 5.000 | 9.000 | 8.000 | 5.000 | 6.000 |
| 5 | 8.000 | 6.000 | 7.000 | 8.000 | 5.000 | 9.000 | 7.000 | 5.000 | 6.000 | 4.000 |

A oferta total é **5,0 t**, igual à demanda total de **5,0 t**.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

print("Gurobi carregado com sucesso.")

In [ ]:
F = [1, 2, 3]
S = [1, 2, 3, 4, 5]
C = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Dados de oferta e demanda do modelo: toneladas
oferta = {1: 1.0, 2: 2.0, 3: 2.0}
demanda = {consumidor: 0.5 for consumidor in C}

# Capacidade de cada silo e limite de cada rota silo -> consumidor
capacidade_silo = {silo: 1.0 for silo in S}
limite_rota = {(silo, consumidor): 0.5 for silo in S for consumidor in C}

print("Oferta total:", sum(oferta.values()), "toneladas")
print("Demanda total:", sum(demanda.values()), "toneladas")
print("Capacidade de cada silo:", capacidade_silo[1], "tonelada")
print("Limite por rota silo-consumidor:", limite_rota[1, 1], "tonelada")

# Custos por tonelada: Fazenda -> Silo (R$/t)
custo_FS = {
    1: {1: 9000, 2: 13000, 3: 8000, 4: 11000, 5: 7000},
    2: {1: 12000, 2: 8000, 3: 10000, 4: 6000, 5: 11000},
    3: {1: 10000, 2: 11000, 3: 7000, 4: 9000, 5: 8000}
}

# Custos por tonelada: Silo -> Consumidor (R$/t)
custo_SC = {
    1: {1: 7000, 2: 5000, 3: 9000, 4: 6000, 5: 8000, 6: 7000, 7: 10000, 8: 6000, 9: 8000, 10: 9000},
    2: {1: 6000, 2: 8000, 3: 5000, 4: 9000, 5: 7000, 6: 6000, 7: 8000, 8: 10000, 9: 6000, 10: 7000},
    3: {1: 9000, 2: 7000, 3: 6000, 4: 5000, 5: 9000, 6: 8000, 7: 6000, 8: 7000, 9: 10000, 10: 5000},
    4: {1: 5000, 2: 9000, 3: 8000, 4: 7000, 5: 6000, 6: 5000, 7: 9000, 8: 8000, 9: 5000, 10: 6000},
    5: {1: 8000, 2: 6000, 3: 7000, 4: 8000, 5: 5000, 6: 9000, 7: 7000, 8: 5000, 9: 6000, 10: 4000}
}

print("")
print("Custos Fazenda -> Silo (R$/t)")
for fazenda in F:
    print(fazenda, ":", [custo_FS[fazenda][silo] for silo in S])

print()
print("Custos Silo -> Consumidor (R$/t)")
for silo in S:
    print(silo, ":", [custo_SC[silo][consumidor] for consumidor in C])

Oferta total: 5.0 toneladas
Demanda total: 5.0 toneladas
Capacidade de cada silo: 1.0 tonelada
Limite por rota silo-consumidor: 0.5 tonelada
Custos Fazenda -> Silo (R$/t)
1 : [9000, 13000, 8000, 11000, 7000]
2 : [12000, 8000, 10000, 6000, 11000]
3 : [10000, 11000, 7000, 9000, 8000]

Custos Silo -> Consumidor (R$/t)
1 : [7000, 5000, 9000, 6000, 8000, 7000, 10000, 6000, 8000, 9000]
2 : [6000, 8000, 5000, 9000, 7000, 6000, 8000, 10000, 6000, 7000]
3 : [9000, 7000, 6000, 5000, 9000, 8000, 6000, 7000, 10000, 5000]
4 : [5000, 9000, 8000, 7000, 6000, 5000, 9000, 8000, 5000, 6000]
5 : [8000, 6000, 7000, 8000, 5000, 9000, 7000, 5000, 6000, 4000]


## 3. Estrutura da rede

O fluxo percorre obrigatoriamente as três camadas: fazendas, silos e consumidores.

```mermaid
flowchart LR
    subgraph L1["CAMADA 1 | FAZENDAS"]
        direction TB
        fazenda1["1<br>Oferta: 1 t"]
        fazenda2["2<br>Oferta: 2 t"]
        fazenda3["3<br>Oferta: 2 t"]
    end

    subgraph L2["CAMADA 2 | SILOS"]
        direction TB
        silo1["1<br>Capacidade: 1 t"]
        silo2["2<br>Capacidade: 1 t"]
        silo3["3<br>Capacidade: 1 t"]
        silo4["4<br>Capacidade: 1 t"]
        silo5["5<br>Capacidade: 1 t"]
    end

    subgraph L3["CAMADA 3 | CONSUMIDORES"]
        direction TB
        consumidores["1 ... 10<br>Demanda: 0,5 t cada"]
    end

    fazenda1 --> silo1
    fazenda1 --> silo2
    fazenda1 --> silo3
    fazenda1 --> silo4
    fazenda1 --> silo5
    fazenda2 --> silo1
    fazenda2 --> silo2
    fazenda2 --> silo3
    fazenda2 --> silo4
    fazenda2 --> silo5
    fazenda3 --> silo1
    fazenda3 --> silo2
    fazenda3 --> silo3
    fazenda3 --> silo4
    fazenda3 --> silo5

    silo1 --> consumidores
    silo2 --> consumidores
    silo3 --> consumidores
    silo4 --> consumidores
    silo5 --> consumidores

    classDef fazenda fill:#dbeafe,stroke:#2563eb,stroke-width:2px,color:#111827
    classDef silo fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#111827
    classDef consumidor fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#111827
    class fazenda1,fazenda2,fazenda3 fazenda
    class silo1,silo2,silo3,silo4,silo5 silo
    class consumidores consumidor
```

Cada seta representa uma rota permitida. Os nós de cada camada são identificados apenas por números: fazendas $1,2,3$, silos $1,2,3,4,5$ e consumidores $1,\ldots,10$.

## 4. Modelo matemático

### Conjuntos e parâmetros

- $F = \{1,2,3\}$: conjunto de fazendas;
- $S = \{1,2,3,4,5\}$: conjunto de silos;
- $C = \{1,\ldots,10\}$: conjunto de consumidores;
- $a_f$: oferta da fazenda $f$, com $a_1=1$, $a_2=2$ e $a_3=2$ t;
- $d_c$: demanda do consumidor $c$, com $d_c=0{,}5$ t para todo $c\in C$;
- $cap_s$: capacidade do silo $s$, com $cap_s=1$ t para todo $s\in S$;
- $u_{sc}$: limite da rota silo-consumidor, com $u_{sc}=0{,}5$ t para todo $s\in S$ e $c\in C$;
- $c^{FS}_{fs}$: custo da rota fazenda-silo, armazenado em `custo_FS` (R$/t);
- $c^{SC}_{sc}$: custo da rota silo-consumidor, armazenado em `custo_SC` (R$/t).

A oferta total é $1+2+2=5$ t e a demanda total é $10\times0{,}5=5$ t.

### Variáveis de decisão

- $x_{fs}\geq0$: quantidade transportada da fazenda $f$ para o silo $s$;
- $y_{sc}\geq0$: quantidade transportada do silo $s$ para o consumidor $c$.

No código, a não negatividade é garantida por `lb=0` nas variáveis `x` e `y`.

### Função objetivo

$$
\min Z =
\underbrace{\sum_{f\in F}\sum_{s\in S} c^{FS}_{fs}x_{fs}}_{\text{fazenda }\rightarrow\text{silo}}
+
\underbrace{\sum_{s\in S}\sum_{c\in C} c^{SC}_{sc}y_{sc}}_{\text{silo }\rightarrow\text{consumidor}}
$$

No código, esta expressão é implementada por:

```python
model.setObjective(
    gp.quicksum(custo_FS[f][silo] * x[f, silo] for f in F for silo in S)
    + gp.quicksum(custo_SC[silo][c] * y[silo, c] for silo in S for c in C),
    GRB.MINIMIZE
)
```

### Restrições implementadas

**1. Oferta de cada fazenda**

$$
\sum_{s\in S}x_{fs}=a_f \qquad \forall f\in F
$$


**2. Conservação de fluxo em cada silo**

$$
\sum_{f\in F}x_{fs}=\sum_{c\in C}y_{sc} \qquad \forall s\in S
$$

**3. Capacidade máxima de cada silo**

$$
\sum_{f\in F}x_{fs}\leq cap_s=1 \qquad \forall s\in S
$$

**4. Demanda de cada consumidor**

$$
\sum_{s\in S}y_{sc}=d_c=0{,}5 \qquad \forall c\in C
$$

**5. Limite de cada rota silo-consumidor**

$$
y_{sc}\leq u_{sc}=0{,}5 \qquad \forall s\in S,\ c\in C
$$

**6. Não negatividade**

$$
x_{fs}\geq0,\qquad y_{sc}\geq0
$$

Todas as restrições acima são criadas na célula de implementação do modelo, usando os dicionários de dados definidos anteriormente.

In [ ]:
model = gp.Model("Transbordo_3_Camadas_Milho")

# Variáveis de decisão: fazenda -> silo e silo -> consumidor
x = model.addVars(F, S, lb=0, vtype=GRB.CONTINUOUS, name="x")
y = model.addVars(S, C, lb=0, vtype=GRB.CONTINUOUS, name="y")

# Função objetivo: custos das duas camadas
model.setObjective(
    gp.quicksum(custo_FS[f][silo] * x[f, silo] for f in F for silo in S)
    + gp.quicksum(custo_SC[silo][c] * y[silo, c] for silo in S for c in C),
    GRB.MINIMIZE
)

# Oferta das fazendas
for f in F:
    model.addConstr(
        gp.quicksum(x[f, silo] for silo in S) == oferta[f],
        name=f"Oferta_{f}"
    )

# Conservação de fluxo e capacidade dos silos
for silo in S:
    entrada = gp.quicksum(x[f, silo] for f in F)
    saida = gp.quicksum(y[silo, c] for c in C)
    model.addConstr(entrada == saida, name=f"Conservacao_{silo}")
    model.addConstr(entrada <= capacidade_silo[silo], name=f"Capacidade_{silo}")

# Demanda dos consumidores e limite das rotas
for c in C:
    model.addConstr(
        gp.quicksum(y[silo, c] for silo in S) == demanda[c],
        name=f"Demanda_{c}"
    )

for silo in S:
    for c in C:
        model.addConstr(
            y[silo, c] <= limite_rota[silo, c],
            name=f"LimiteRota_{silo}_{c}"
        )

model.optimize()

print("-----")
if model.Status == GRB.OPTIMAL:
    print("Status: solução ótima encontrada")
    print("Custo total mínimo: R$ {:.2f}".format(model.ObjVal))
else:
    print("Status do modelo:", model.Status)


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 73 rows, 65 columns and 195 nonzeros (Min)
Model fingerprint: 0x6249bbe7
Model has 65 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+03, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e-01, 2e+00]

Presolve removed 51 rows and 0 columns
Presolve time: 0.00s
Presolved: 22 rows, 65 columns, 132 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.8000000e+04   6.496000e+00   0.000000e+00      0s
      10    6.4000000e+04   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.00 seconds (0.00 work units)
Optimal objective  6.400000000e+04


## 5. Caminhos ótimos da solução

In [32]:
print("{:<24} {:>12}".format("Caminho", "Toneladas"))
print("-" * 38)

for f in F:
    for silo in S:
        for c in C:
            fluxo_fazenda_silo = x[f, silo].X
            fluxo_silo_consumidor = y[silo, c].X
            if fluxo_fazenda_silo > 1e-6 and fluxo_silo_consumidor > 1e-6:
                print("{:<24} {:>12.2f}".format(
                    f"{f} -> {silo} -> {c}",
                    fluxo_silo_consumidor
                ))

Caminho                     Toneladas
--------------------------------------
1 -> 5 -> 5                      0.50
1 -> 5 -> 10                     0.50
2 -> 2 -> 1                      0.50
2 -> 2 -> 3                      0.50
2 -> 4 -> 6                      0.50
2 -> 4 -> 9                      0.50
3 -> 1 -> 2                      0.50
3 -> 1 -> 8                      0.50
3 -> 3 -> 4                      0.50
3 -> 3 -> 7                      0.50


## 6. Verificação das ofertas/demandas

1. A soma dos fluxos que saem de cada fazenda deve ser igual à oferta disponível.
2. Para cada silo, a quantidade que entra deve ser igual à quantidade que sai e não pode exceder 1 t.
3. Cada consumidor deve receber exatamente sua demanda.

In [ ]:
#verificação ofertas
print("{:<12} {:>15} {:>15} {:>15}".format(
    "Fazenda", "Oferta (t)", "Enviado (t)", "Diferença"
))
print("-" * 62)

for f in F:
    enviado = sum(x[f, silo].X for silo in S)
    print("{:<12} {:>15.2f} {:>15.2f} {:>15.6f}".format(
        f, oferta[f], enviado, enviado - oferta[f]
    ))

#verificação transbordo
print("{:<12} {:>15} {:>15} {:>15} {:>15}".format(
    "Silo", "Entrada (t)", "Saída (t)", "Capacidade", "Diferença"
))
print("-" * 80)

for silo in S:
    entrada = sum(x[f, silo].X for f in F)
    saida = sum(y[silo, c].X for c in C)
    print("{:<12} {:>15.2f} {:>15.2f} {:>15.2f} {:>15.6f}".format(
        silo, entrada, saida, capacidade_silo[silo], entrada - saida
    ))

#verificação demandas
print("{:<14} {:>15} {:>15} {:>15}".format(
    "Consumidor", "Demanda (t)", "Recebido (t)", "Diferença"
))
print("-" * 65)

for c in C:
    recebido = sum(y[silo, c].X for silo in S)
    print("{:<14} {:>15.2f} {:>15.2f} {:>15.6f}".format(
        c, demanda[c], recebido, recebido - demanda[c]
    ))

maior_excesso_rota = max(
    y[silo, c].X - limite_rota[silo, c]
    for silo in S for c in C
)
print("Maior excesso em limite de rota: {:.6f} t".format(maior_excesso_rota))

Fazenda           Oferta (t)     Enviado (t)       Diferença
--------------------------------------------------------------
1                       1.00            1.00        0.000000
2                       2.00            2.00        0.000000
3                       2.00            2.00        0.000000


## 7. Resumo dos custos

In [31]:
total_t = sum(demanda.values())
custo_fs = sum(
    custo_FS[f][silo] * x[f, silo].X
    for f in F for silo in S
)
custo_sc = sum(
    custo_SC[silo][c] * y[silo, c].X
    for silo in S for c in C
)
custo_total = custo_fs + custo_sc

print("CUSTOS DA SOLUÇÃO ÓTIMA")
print("-" * 45)
print("Fazenda -> Silo          : R$ {:,.2f}".format(custo_fs))
print("Silo -> Consumidor       : R$ {:,.2f}".format(custo_sc))
print("Custo total              : R$ {:,.2f}".format(custo_total))
print("Toneladas transportadas  : {:,.2f} t".format(total_t))
print("Custo médio por tonelada : R$ {:,.2f}/t".format(custo_total / total_t))

CUSTOS DA SOLUÇÃO ÓTIMA
---------------------------------------------
Fazenda -> Silo          : R$ 38,000.00
Silo -> Consumidor       : R$ 26,000.00
Custo total              : R$ 64,000.00
Toneladas transportadas  : 5.00 t
Custo médio por tonelada : R$ 12,800.00/t


## 8. Análise da solução

O modelo representa uma rede com três camadas: as fazendas fornecem o milho, os silos funcionam como depósitos intermediários e os consumidores recebem o produto. Essa estrutura é uma extensão do problema de transbordo apresentado na Aula 2, pois agora existem duas etapas de transporte e três conjuntos de nós.

A solução encontrada pelo Gurobi tem custo total de **R$ 64.000,00**. Desse valor, **R$ 38.000,00** correspondem ao transporte das fazendas para os silos e **R$ 26.000,00** ao transporte dos silos para os consumidores.

As três fazendas utilizam toda a oferta disponível: a fazenda 1 envia 1 t, a fazenda 2 envia 2 t e a fazenda 3 envia 2 t. Como a capacidade de cada silo é de 1 t e a oferta total é de 5 t, os cinco silos recebem exatamente 1 t. A conservação de fluxo garante que essa mesma quantidade seja enviada aos consumidores.

Cada consumidor recebe exatamente 0,5 t. Os caminhos com fluxo positivo são apresentados na tabela anterior, e cada um transporta 0,5 t. O limite máximo por rota também é respeitado, pois nenhuma ligação entre silo e consumidor transporta mais do que essa quantidade.

Portanto, a solução atende integralmente às ofertas e demandas, respeita as capacidades dos silos e os limites das rotas e minimiza o custo total de transporte para os dados considerados. Como os custos e as quantidades são hipotéticos, o valor obtido representa o melhor resultado para este cenário específico.